# Control vs Ketamine vs Psilocybin (AdEx Whole-Brain)

This notebook runs strict-parity AdEx whole-brain simulations (Zerlaut family), computes **LZc** and a **PCI (Casali-like)** score, and saves outputs in an intuitive directory layout.

In [1]:
from pathlib import Path
import sys
import numpy as np

# Ensure notebook uses local source tree for the latest toolkit code
ROOT = Path.cwd().resolve().parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from tvbtoolkit import (
    WholeBrainConfig, OutputConfig, ConditionSpec, run_condition_batch,
    SingleRegionConfig, run_single_region_simulation,
    plot_example_timeseries, plot_single_region_timeseries, plot_timeseries, plot_metric_summary,
    detect_system_specs, recommend_parallel_workers
)

# Guard against stale installs
if 'monitor_mode' not in WholeBrainConfig.__dataclass_fields__:
    raise RuntimeError(
        'Stale tvbtoolkit import detected: WholeBrainConfig has no monitor_mode. '
        'Restart kernel and reinstall with: pip install -e /Users/borjan/CNRS/projects/TVBToolkit'
    )
CONNECTIVITY_68 = ROOT / 'data' / 'connectivity' / 'connectivity_68.zip'
if not CONNECTIVITY_68.exists():
    raise FileNotFoundError(f'Connectivity file not found: {CONNECTIVITY_68}')

## System + Parallelism Info

This notebook auto-detects hardware specs and recommends a safe CPU worker count.

Use `FAST_MODE=True` for fast iteration, then set `FAST_MODE=False` for final-quality runs.

Note: TVB/Brian2 execution here is CPU-based; integrated GPU is not used by default.

Batch runs default to the **temporal average monitor** for speed and smaller outputs.

In [ ]:
specs = detect_system_specs()
print(specs)

FAST_MODE = True  # Set False for final-quality outputs

FAST_CFG = {
    'simulation_length_ms': 4000.0,
    'n_seeds': 1,
    'save_timeseries': False,
    'n_jobs_cap': 5,
    'monitor_mode': 'temporal_average',
    'temporal_average_period_ms': 1.0,
}

FINAL_CFG = {
    'simulation_length_ms': 5000.0,
    'n_seeds': 5,
    'save_timeseries': True,
    'n_jobs_cap': None,
    'monitor_mode': 'temporal_average',
    'temporal_average_period_ms': 1.0,
}

RUN_CFG = FAST_CFG if FAST_MODE else FINAL_CFG

N_JOBS = recommend_parallel_workers(task='whole_brain_tvb')
if RUN_CFG['n_jobs_cap'] is not None:
    N_JOBS = min(N_JOBS, RUN_CFG['n_jobs_cap'])

print('Mode:', 'FAST' if FAST_MODE else 'FINAL')
print('Run config:', RUN_CFG)
print('Workers used:', N_JOBS)

In [ ]:
# Output layout
out = OutputConfig(Path('./outputs/control_ketamine_psilocybin'))
out

## 1) Base AdEx whole-brain config

Strict parity defaults are enabled via `model_family='adex_zerlaut'`.

In [ ]:
base_cfg = WholeBrainConfig(
    model_family='adex_zerlaut',
    zerlaut_matteo=False,
    zerlaut_gk_gna=False,
    zerlaut_order=2,  # second-order: matches TVBSim default
    stochastic_integrator=True,
    monitor_mode=RUN_CFG['monitor_mode'],
    temporal_average_period_ms=RUN_CFG['temporal_average_period_ms'],
    simulation_length_ms=RUN_CFG['simulation_length_ms'],
    dt_ms=0.25,
    conduction_speed=4.0,
    coupling_strength=0.3,
    connectivity_zip=str(CONNECTIVITY_68),
    parameter_overrides={
        'parameter_model': {
            'T': 40.0,
            'P_e': [-0.0498, 0.00506, -0.025, 0.0014, -0.00041, 0.0105, -0.036, 0.0074, 0.0012, -0.0407],
            'P_i': [-0.0514, 0.004, -0.0083, 0.0002, -0.0005, 0.0014, -0.0146, 0.0045, 0.0028, -0.0153],
            'initial_condition': {
                'E': [0.004, 0.004], 'I': [0.010, 0.010],
                'C_ee': [0.0, 0.0], 'C_ei': [0.0, 0.0], 'C_ii': [0.0, 0.0],
                'W_e': [50.0, 50.0], 'W_i': [0.0, 0.0],
                'noise': [0.0, 0.0],
            },
        },
        'parameter_simulation': {'save_time': 1000.0},
    },
)
base_cfg

## 2) Condition definitions

These are parity-oriented condition presets that you can tune further.

In [ ]:
conditions = [
    ConditionSpec(
        name='control',
        description='Baseline awake-like condition',
        parameter_overrides={}
    ),
    ConditionSpec(
        name='ketamine',
        description='Ketamine-like shift (higher adaptation / altered synaptic time constants)',
        parameter_overrides={
            'parameter_model': {
                'b_e': 25.0,
                'tau_e_e': 4.5,
                'tau_e_i': 4.5,
            }
        },
    ),
    ConditionSpec(
        name='psilocybin',
        description='Psilocybin-like shift using gK/gNa mode',
        parameter_overrides={
            'parameter_model': {
                'gK_gNa': True,
                'g_K_e': 5.8,
                'g_K_i': 5.8,
                'E_L_e': -61.0,
                'E_L_i': -63.0,
            }
        },
    ),
]

seeds = list(range(RUN_CFG['n_seeds']))
conditions

## 3) Run simulations + compute LZc and PCI (Casali-like)

Outputs saved automatically:
- `outputs/control_ketamine_psilocybin/simulations/<condition>/seed_XXX.npz`
- `outputs/control_ketamine_psilocybin/metrics/<condition>_metrics.npz`

In [ ]:
metrics_by_condition = run_condition_batch(
    base_cfg=base_cfg,
    conditions=conditions,
    seeds=seeds,
    output=out,
    post_stim_window=300,
    save_timeseries=RUN_CFG['save_timeseries'],
    n_jobs=N_JOBS,
    use_processes=True,
    show_progress=True,
    monitor_mode_default='temporal_average',
    temporal_average_period_ms=RUN_CFG['temporal_average_period_ms'],
)
list(metrics_by_condition.keys())

## 3b) Monitor Sampling Benchmark

Quick check of expected sample-count reduction from raw to temporal-average monitoring.

In [ ]:
n_samples_raw = int(base_cfg.simulation_length_ms / base_cfg.dt_ms)
n_samples_tavg = int(base_cfg.simulation_length_ms / RUN_CFG['temporal_average_period_ms'])
reduction = n_samples_raw / max(n_samples_tavg, 1)
print('Raw samples per run:        ', n_samples_raw)
print('Temporal-avg samples per run:', n_samples_tavg)
print('Expected reduction factor:   ', f'{reduction:.1f}x')

if RUN_CFG['save_timeseries']:
    import os
    sim_dir = out.simulations_dir
    total_mb = 0.0
    for root, _, files in os.walk(sim_dir):
        for fn in files:
            if fn.endswith('.npz'):
                total_mb += os.path.getsize(os.path.join(root, fn)) / (1024**2)
    print('Saved timeseries size so far (MB):', round(total_mb, 2))

## 4) Create and save publication-style figures

In [ ]:
fig1 = plot_timeseries(
    metrics_by_condition,
    source='whole_brain',
    max_regions=6,
    trim_first_ms=1000,
    save_path=out.figures_dir / 'condition_timeseries.png',
)
fig2 = plot_metric_summary(
    metrics_by_condition,
    save_path=out.figures_dir / 'lzc_pci_summary.png',
)
print('Saved:', out.figures_dir / 'condition_timeseries.png')
print('Saved:', out.figures_dir / 'lzc_pci_summary.png')

## 5) Single-Region Extension: Spiking and Reduced Paths

This extension plots firing rates from single-region simulations using the same visual language as whole-brain figures.
- `source='single_region_spiking'`: plots rates + spike rasters (if available)
- `source='single_region_reduced'`: plots rates only

In [ ]:
single_cfg = SingleRegionConfig(
    duration_ms=RUN_CFG['simulation_length_ms'],
    dt_ms=base_cfg.dt_ms,
    record_spikes=True,
)
single_res = run_single_region_simulation(single_cfg, seed_value=seeds[0])

fig_sr_spk = plot_single_region_timeseries(
    single_res,
    source='single_region_spiking',
    include_spikes=True,
    save_path=out.figures_dir / 'single_region_spiking_timeseries.png',
)

single_reduced = {
    'time_ms': single_res.time_ms,
    'exc_rate_hz': single_res.exc_rate_hz,
    'inh_rate_hz': single_res.inh_rate_hz,
}
fig_sr_red = plot_timeseries(
    single_reduced,
    source='single_region_reduced',
    include_spikes=False,
    save_path=out.figures_dir / 'single_region_reduced_timeseries.png',
)

print('Saved:', out.figures_dir / 'single_region_spiking_timeseries.png')
print('Saved:', out.figures_dir / 'single_region_reduced_timeseries.png')

In [ ]:
# Quick numeric summary
for cond, vals in metrics_by_condition.items():
    print(
        cond,
        'LZc mean=', float(np.mean(vals['lzc'])),
        'PCI (Casali-like) mean=', float(np.mean(vals['pci_casali_like']))
    )